<a href="https://colab.research.google.com/github/chaeeunmin/chaeeunmin-High-school-students-breakfast-skipping-rate-and-obesity-rate-relationship/blob/main/10_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# [Cell 1] 구글 드라이브 연결 및 필요 라이브러리 불러오기
from google.colab import drive
import pandas as pd
import numpy as np

# 구글 드라이브 마운트
drive.mount('/content/drive')

# 2025년 청소년건강행태조사 원시자료 불러오기
df = pd.read_sas('/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat')
print(f"전체 원시자료 행 수: {len(df)}명")

# ----------------------------------------------------------------------

# [Cell 2] 고등학생 도메인 추출, 아침 결식 및 비만 변수 생성 (B-1)
# GRADE 4, 5, 6: 고등학교 1, 2, 3학년
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)

# 아침식사 결식 변수 (F_BR: 1~3은 최근 7일 중 5일 이상 결식)
df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
                         np.where(df['F_BR'].notnull(), 0, np.nan))

# BMI 산출 (HT: cm, WT: kg)
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)

# 비만 변수 생성 (BMI 25 이상을 비만으로 분류하여 KeyError 예방)
df['obesity'] = np.where(df['bmi'] >= 25, 1,
                         np.where(df['bmi'].notnull(), 0, np.nan))

# 유효 인원 표식 (valid_sample: 고등학생 도메인 & 아침결식/BMI 유효응답자)
df['valid_sample'] = (df['domain_hs'] == 1) & df['bf_skip'].notnull() & df['bmi'].notnull()
valid_df = df[df['valid_sample'] == True].copy()

print(f"최종 유효 분석 대상자 수: {len(valid_df)}명")

# ----------------------------------------------------------------------

# [Cell 3] 복합표본 설계를 반영한 가중 백분율 및 평균 산출 (B-2)
def get_weighted_pct(data, var_name, weight_col='W'):
    grouped = data.groupby(var_name)[weight_col].sum()
    total_w = grouped.sum()
    return (grouped / total_w) * 100

print("\n=== 성별 가중 백분율 (%) ===")
print(get_weighted_pct(valid_df, 'SEX'))

print("\n=== 비만 여부 가중 백분율 (%) ===")
print(get_weighted_pct(valid_df, 'obesity'))

# BMI 가중 평균 및 표준오차(SE) 간이 계산
weighted_bmi_mean = np.average(valid_df['bmi'], weights=valid_df['W'])
print(f"\nBMI 가중 평균: {weighted_bmi_mean:.2f} kg/m²")

# ----------------------------------------------------------------------

# [Cell 4] 비가중 대 가중 아침 결식률 비교 (B-3)
unweighted_skip_rate = (valid_df['bf_skip'].sum() / len(valid_df)) * 100
weighted_skip_rate = (valid_df.groupby('bf_skip')['W'].sum()[1] / valid_df['W'].sum()) * 100

print("\n=== B-3 비가중 대 가중 비교 ===")
print(f"비가중 아침 결식률: {unweighted_skip_rate:.2f}%")
print(f"가중 아침 결식률: {weighted_skip_rate:.2f}%")

Mounted at /content/drive
전체 원시자료 행 수: 54170명
최종 유효 분석 대상자 수: 25077명

=== 성별 가중 백분율 (%) ===
SEX
1.0    51.551269
2.0    48.448731
Name: W, dtype: float64

=== 비만 여부 가중 백분율 (%) ===
obesity
0.0    79.943783
1.0    20.056217
Name: W, dtype: float64

BMI 가중 평균: 22.18 kg/m²

=== B-3 비가중 대 가중 비교 ===
비가중 아침 결식률: 45.36%
가중 아침 결식률: 45.68%


/tmp/ipykernel_11613/910459220.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
/tmp/ipykernel_11613/910459220.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
/tmp/ipykernel_11613/910459220.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instea

In [3]:
# ==========================================
# 분석 대상자 선정 흐름도 단계별 인원수 출력
# ==========================================

# 1단계: 전체 조사 표본
print(f"1단계 (전체 조사 표본): N = {len(df):,}")

# 2단계: 고등학생 도메인 추출 (학년 GRADE가 고등학교 1~3학년인 경우)
# (보통 청소년건강행태조사에서 학년 변수 기준 확인)
hs_domain_df = df[df['GRADE'].isin([4, 5, 6])].copy() # 4, 5, 6학년 = 고1, 고2, 고3
excluded_middle = len(df) - len(hs_domain_df)
print(f"2단계 (고등학생 도메인 추출): n = {len(hs_domain_df):,}명 선택 (제외된 중학생 등: {excluded_middle:,}명)")

# 3단계: 아침식사 변수 확인 (결측치 및 비해당 확인)
br_missing = hs_domain_df['F_BR'].isna().sum()
print(f"3단계 (아침식사 변수 확인): 결측치/무응답 인원 = {br_missing:,}명")

# 4단계: 신장·체중 및 필수변수 결측치 및 이상치 제외
# (만약 이전 문서의 1,532명과 현재 766명 중 코드 상으로 실제로 필터링되는 정확한 숫자가 출력되도록 확인)
before_cleaning = len(hs_domain_df)
# 신장(HT), 체중(WT), 아침식사(F_BR) 등의 결측치 제거 조건
cleaned_df = hs_domain_df.dropna(subset=['HT', 'WT', 'F_BR']).copy()
excluded_hw = before_cleaning - len(cleaned_df)
print(f"4단계 (신장·체중 결측치 및 이상치 제외): 제외 인원 = {excluded_hw:,}명")

# 5단계: 최종 분석 유효 인원 및 결식군/비결식군 분류
# 결식군: 주 5회 이상 결식 (예: 0~2일), 비결식군: 3~7일
cleaned_df['skip_group'] = cleaned_df['F_BR'].apply(lambda x: 1 if x in [1, 2, 3] else 0)
skip_n = (cleaned_df['skip_group'] == 1).sum()
non_skip_n = (cleaned_df['skip_group'] == 0).sum()

print(f"5단계 (최종 분석 유효 인원): N = {len(cleaned_df):,}")
print(f"  - 아침식사 결식군 (0–2일): n = {skip_n:,}명")
print(f"  - 아침식사 비결식군 (3–7일): n = {non_skip_n:,}명")


1단계 (전체 조사 표본): N = 54,170
2단계 (고등학생 도메인 추출): n = 25,843명 선택 (제외된 중학생 등: 28,327명)
3단계 (아침식사 변수 확인): 결측치/무응답 인원 = 0명
4단계 (신장·체중 결측치 및 이상치 제외): 제외 인원 = 766명
5단계 (최종 분석 유효 인원): N = 25,077
  - 아침식사 결식군 (0–2일): n = 11,376명
  - 아침식사 비결식군 (3–7일): n = 13,701명
